# Jour 1 : Introduction à PyTorch

**Deep Learning IndabaX RDC 2026 · Atelier PyTorch**
Université Nouveaux Horizons (UNH), Lubumbashi · 26 novembre 2026

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/blob/main/notebooks/Jour1_Introduction_PyTorch.ipynb)

Ce notebook suit le parcours officiel **[Learn the Basics](https://docs.pytorch.org/tutorials/beginner/basics/intro.html)** de PyTorch.
Nous l'avons adapté en français, avec des schémas, des exemples en langues africaines et des exercices.
Le texte est en français. Les commentaires du code sont en anglais, comme dans la documentation officielle.

### Objectifs de la journée

À la fin de ce notebook, vous savez :

1. créer et manipuler des **tenseurs** ;
2. faire vos calculs sur le **GPU** de Google Colab ;
3. expliquer ce que fait **autograd** (le calcul automatique des gradients) ;
4. charger des données avec **`Dataset`** et **`DataLoader`** ;
5. construire un réseau de neurones avec **`nn.Module`** ;
6. écrire la **boucle d'entraînement** et évaluer le modèle ;
7. **sauvegarder** et **charger** un modèle ;
8. expliquer ce que font un **tokenizer** et un **modèle de langage** (préparation du Jour 2).

### Programme

<p align="center"><img src="https://raw.githubusercontent.com/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/main/images/01_carte_atelier.png" alt="Carte de l'atelier du jour 1" width="900"></p>

| Partie | Sujet | Durée indicative | Tutoriel officiel |
|---|---|---|---|
| 0 | Mise en place de Colab | 5 min | - |
| 1 | Les tenseurs | 15 min | [Tensors](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html) |
| 2 | Le GPU | 5 min | [Tensors](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html) |
| 3 | Autograd et descente de gradient | 15 min | [Autograd](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html) |
| 4 | Les données | 10 min | [Datasets & DataLoaders](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html), [Transforms](https://docs.pytorch.org/tutorials/beginner/basics/transforms_tutorial.html) |
| 5 | Construire le modèle | 10 min | [Build Model](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html) |
| 6 | La boucle d'entraînement | 15 min | [Optimization](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) |
| 7 | Sauvegarder et charger | 5 min | [Save & Load](https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html) |
| 8 | Pont vers le Jour 2 : les modèles de langage | 10 min | - |

### Avant de commencer (Google Colab)

1. Connectez-vous avec votre compte Google.
2. Menu **Exécution → Modifier le type d'exécution** (*Runtime → Change runtime type*) → choisissez **GPU T4** → **Enregistrer**.
3. Pour exécuter une cellule : cliquez dedans, puis **Maj + Entrée** (*Shift + Enter*).
4. Exécutez les cellules **dans l'ordre**, de haut en bas.

> Pas de GPU disponible ? Pas de problème. Tout le notebook fonctionne aussi sur CPU. Il est seulement plus lent.

**Prérequis :** des bases de Python (variables, listes, fonctions, boucles `for`). Aucune expérience en apprentissage profond n'est nécessaire.

## 0. Mise en place

Cette cellule importe les bibliothèques, affiche les versions et choisit l'appareil de calcul (*device*) : le GPU s'il existe, sinon le CPU.

In [ ]:
# Import the libraries used in this notebook
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from torch import nn

print(f"PyTorch version     : {torch.__version__}")
print(f"torchvision version : {torchvision.__version__}")

# Pick the best available device: the GPU (accelerator) if present, else the CPU.
# This is the same code as in the official PyTorch Quickstart.
if hasattr(torch, "accelerator") and torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator().type
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device              : {device}")
if device == "cuda":
    print(f"GPU                 : {torch.cuda.get_device_name(0)}")

# Fix the random seed so that everybody gets the same numbers
_ = torch.manual_seed(42)

## 1. Les tenseurs

Un **tenseur** (*tensor*) est un tableau de nombres à N dimensions. C'est la structure de données de base de PyTorch.
Les entrées d'un modèle, ses sorties et ses paramètres (ses poids) sont tous des tenseurs.

Un tenseur ressemble beaucoup à un tableau NumPy (`ndarray`), avec deux différences importantes :

- un tenseur peut être calculé sur un **GPU** ;
- un tenseur peut garder la trace des opérations pour calculer les **gradients** (partie 3).

<p align="center"><img src="https://raw.githubusercontent.com/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/main/images/02_tenseurs_dimensions.png" alt="Les dimensions d'un tenseur : scalaire, vecteur, matrice, image, lot d'images" width="900"></p>

| Nombre de dimensions | Nom | Exemple |
|---|---|---|
| 0 | scalaire | une température : `24.5` |
| 1 | vecteur | les notes d'un élève dans 3 matières |
| 2 | matrice | une image en niveaux de gris (hauteur × largeur) |
| 3 | tenseur 3D | une image en couleur (canaux × hauteur × largeur) |
| 4 | tenseur 4D | un **lot** (*batch*) d'images (lot × canaux × hauteur × largeur) |

Tutoriel officiel : [Tensors](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html)

### 1.1 Créer un tenseur

Il existe plusieurs façons de créer un tenseur.

In [ ]:
# 1) Directly from Python data (a list of lists)
data = [[1, 2], [3, 4]]
x_data = torch.tensor(data)
print("From a list:\n", x_data)

# 2) From a NumPy array
np_array = np.array(data)
x_np = torch.from_numpy(np_array)
print("\nFrom NumPy:\n", x_np)

# 3) From another tensor: same shape, new values
x_ones = torch.ones_like(x_data)                     # keeps the shape and dtype of x_data
x_rand = torch.rand_like(x_data, dtype=torch.float)  # keeps the shape, changes the dtype
print("\nOnes tensor:\n", x_ones)
print("\nRandom tensor:\n", x_rand)

In [ ]:
# 4) From a shape: a tuple that gives the size of each dimension
shape = (2, 3)
rand_tensor = torch.rand(shape)    # uniform random numbers in [0, 1)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random tensor:\n{rand_tensor}\n")
print(f"Ones tensor:\n{ones_tensor}\n")
print(f"Zeros tensor:\n{zeros_tensor}\n")

# Other useful constructors
print("arange :", torch.arange(0, 10, 2))       # 0, 2, 4, 6, 8
print("linspace:", torch.linspace(0, 1, 5))     # 5 numbers from 0 to 1
print("randn  :", torch.randn(3))               # normal distribution (mean 0, std 1)

### 1.2 Les attributs d'un tenseur

Chaque tenseur a trois attributs à connaître :

- `shape` : sa forme (la taille de chaque dimension) ;
- `dtype` : le type des nombres (`float32`, `int64`, ...) ;
- `device` : l'appareil où il est stocké (`cpu` ou `cuda`).

> **Astuce :** 90 % des erreurs des débutants viennent d'une mauvaise `shape`, d'un mauvais `dtype` ou d'un mauvais `device`. En cas d'erreur, affichez ces trois attributs.

In [ ]:
tensor = torch.rand(3, 4)

print(f"Shape of tensor : {tensor.shape}")
print(f"Datatype        : {tensor.dtype}")
print(f"Device          : {tensor.device}")
print(f"Number of dims  : {tensor.ndim}")
print(f"Number of items : {tensor.numel()}")

### 1.3 Indexer et découper (*indexing* et *slicing*)

Comme avec NumPy, on lit et on modifie une partie d'un tenseur avec les crochets `[ ]`.
Le premier indice est la ligne, le deuxième est la colonne. Les indices commencent à **0**.

In [ ]:
tensor = torch.ones(4, 4)
print(f"First row    : {tensor[0]}")
print(f"First column : {tensor[:, 0]}")
print(f"Last column  : {tensor[..., -1]}")

tensor[:, 1] = 0          # set the second column to 0
print(tensor)

### 1.4 Assembler des tenseurs

- `torch.cat` colle des tenseurs **le long d'une dimension qui existe déjà** ;
- `torch.stack` empile des tenseurs **dans une nouvelle dimension**.

In [ ]:
a = torch.zeros(2, 3)
b = torch.ones(2, 3)

print("cat  dim=0:", torch.cat([a, b], dim=0).shape)   # (4, 3): more rows
print("cat  dim=1:", torch.cat([a, b], dim=1).shape)   # (2, 6): more columns
print("stack dim=0:", torch.stack([a, b], dim=0).shape)  # (2, 2, 3): a new first dimension

### 1.5 Les opérations

PyTorch a plus de 1 200 opérations sur les tenseurs : calcul, algèbre linéaire, statistiques, etc.
Voir la [liste complète](https://docs.pytorch.org/docs/stable/torch.html).

Deux multiplications sont différentes, et il faut bien les distinguer :

| Opération | Code | Ce qu'elle fait |
|---|---|---|
| Produit élément par élément | `a * b` | multiplie les nombres qui sont à la même position |
| Produit matriciel | `a @ b` | ligne × colonne (comme en cours d'algèbre). C'est l'opération de base des réseaux de neurones. |

In [ ]:
a = torch.tensor([[1., 2.],
                  [3., 4.]])
b = torch.tensor([[10., 20.],
                  [30., 40.]])

print("Element-wise product a * b:\n", a * b)
print("\nMatrix product a @ b:\n", a @ b)    # same as a.matmul(b) or torch.matmul(a, b)
print("\nTranspose a.T:\n", a.T)

In [ ]:
# Aggregations: reduce many values to fewer values
t = torch.arange(1., 7.).reshape(2, 3)    # [[1, 2, 3], [4, 5, 6]]
print(t)
print("sum of all       :", t.sum())
print("sum of each column (dim=0):", t.sum(dim=0))
print("mean of each row   (dim=1):", t.mean(dim=1))
print("max and its index:", t.max(), t.argmax())

# .item() turns a tensor with one value into a Python number
total = t.sum().item()
print(total, type(total))

**Opérations « en place » (*in-place*)** : une opération qui finit par `_` modifie le tenseur lui-même.
Par exemple, `x.add_(5)` ajoute 5 à `x` sans créer de nouveau tenseur.
Elles économisent de la mémoire, mais elles effacent l'historique utile pour les gradients. Utilisez-les avec prudence.

In [ ]:
x = torch.ones(2, 2)
x.add_(5)       # in-place: x itself changes
print(x)

### 1.6 Changer la forme d'un tenseur

Les réseaux de neurones attendent des formes précises. Il faut souvent **réorganiser** un tenseur sans changer ses valeurs.

- `reshape(...)` : nouvelle forme, même nombre d'éléments (`-1` veut dire « calcule cette taille pour moi ») ;
- `unsqueeze(dim)` : ajoute une dimension de taille 1 (par exemple pour faire un lot d'une seule image) ;
- `squeeze()` : enlève les dimensions de taille 1 ;
- `flatten()` : met tout « à plat » dans une seule dimension.

In [ ]:
image = torch.rand(28, 28)                 # one grey image: height x width
print("image            :", image.shape)
print("flatten          :", image.flatten().shape)          # 784 = 28 * 28
print("reshape(-1)      :", image.reshape(-1).shape)
print("add channel dim  :", image.unsqueeze(0).shape)       # (1, 28, 28)
print("add batch dim    :", image.unsqueeze(0).unsqueeze(0).shape)  # (1, 1, 28, 28)
print("squeeze          :", image.unsqueeze(0).squeeze().shape)

### 1.7 La diffusion (*broadcasting*)

Quand deux tenseurs n'ont pas la même forme, PyTorch « étire » automatiquement les dimensions de taille 1.
C'est pratique, par exemple pour ajouter un biais à chaque ligne d'une matrice.

In [ ]:
col = torch.tensor([[1.], [2.], [3.]])   # shape (3, 1)
row = torch.tensor([[10., 20., 30.]])    # shape (1, 3)
print(col + row)                         # shape (3, 3): each row + each column

### 1.8 Le pont avec NumPy

Un tenseur sur CPU et un tableau NumPy peuvent **partager la même mémoire**.
Si l'un change, l'autre change aussi.

In [ ]:
t = torch.ones(5)
n = t.numpy()           # tensor -> NumPy (shared memory)
t.add_(1)
print(f"t: {t}")
print(f"n: {n}   <- n changed too!")

n = np.ones(5)
t = torch.from_numpy(n) # NumPy -> tensor (shared memory)
np.add(n, 1, out=n)
print(f"t: {t}   <- t changed too!")

### ✍️ Exercice 1 : les tenseurs

1. Créez un tenseur `m` de forme `(3, 4)` qui contient les nombres de 1 à 12.
2. Affichez sa deuxième ligne.
3. Calculez la somme de chaque **colonne**.
4. Calculez `m @ m.T`. Quelle est la forme du résultat ? Pourquoi ?

<details>
<summary><b>Voir la solution</b></summary>

```python
m = torch.arange(1, 13).reshape(3, 4)
print(m[1])            # second row: tensor([5, 6, 7, 8])
print(m.sum(dim=0))    # tensor([15, 18, 21, 24])
print((m @ m.T).shape) # torch.Size([3, 3]): (3, 4) @ (4, 3) -> (3, 3)
```
</details>

In [ ]:
# Your code here (Exercise 1)

## 2. Le GPU

Un **CPU** a quelques cœurs puissants. Il fait bien des tâches variées, une après l'autre.
Un **GPU** a des milliers de petits cœurs. Il fait très bien **le même calcul sur beaucoup de nombres en même temps**,
par exemple un produit de matrices. C'est exactement le travail d'un réseau de neurones.

<p align="center"><img src="https://raw.githubusercontent.com/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/main/images/03_cpu_vs_gpu.png" alt="CPU et GPU : quelques cœurs puissants ou des milliers de petits cœurs" width="900"></p>

Par défaut, un tenseur est créé sur le CPU. On le déplace avec `.to(device)`.

> **Règle :** tous les tenseurs d'un même calcul doivent être sur le **même appareil**.

In [ ]:
tensor = torch.rand(3, 4)
print(f"Before: {tensor.device}")

tensor = tensor.to(device)          # move it to the GPU (if there is one)
print(f"After : {tensor.device}")

# You can also create a tensor directly on the device
direct = torch.zeros(2, 2, device=device)
print(f"Created on: {direct.device}")

Mesurons la différence de vitesse sur un grand produit de matrices (4096 × 4096).

In [ ]:
def time_matmul(dev, n=4096):
    """Return the time (seconds) of one n x n matrix product on device dev."""
    a = torch.rand(n, n, device=dev)
    b = torch.rand(n, n, device=dev)
    _ = a @ b                      # warm-up (the first call is slower)
    if dev == "cuda":
        torch.cuda.synchronize()   # the GPU works asynchronously: wait for it
    start = time.perf_counter()
    _ = a @ b
    if dev == "cuda":
        torch.cuda.synchronize()
    return time.perf_counter() - start

cpu_time = time_matmul("cpu")
print(f"CPU: {cpu_time * 1000:8.1f} ms")
if device == "cuda":
    gpu_time = time_matmul("cuda")
    print(f"GPU: {gpu_time * 1000:8.1f} ms  ->  about {cpu_time / gpu_time:.0f}x faster")
else:
    print("No GPU here. In Colab: Runtime -> Change runtime type -> T4 GPU.")

Et si on mélange les appareils ? PyTorch refuse le calcul et donne une erreur claire.

In [ ]:
if device == "cuda":
    a = torch.rand(2, 2)                 # on the CPU
    b = torch.rand(2, 2, device=device)  # on the GPU
    try:
        a + b
    except RuntimeError as err:
        print("RuntimeError:", str(err)[:120], "...")
    print("Fix:", (a.to(device) + b).device)
else:
    print("This demo needs a GPU.")

## 3. Autograd : calculer les gradients automatiquement

Pour apprendre, un réseau de neurones corrige ses poids pour réduire son **erreur** (la **perte**, *loss*).
Pour savoir dans quelle direction corriger chaque poids, il faut la **dérivée** de la perte par rapport à ce poids : le **gradient**.

Le moteur `torch.autograd` de PyTorch calcule ces gradients **automatiquement**.

Tutoriel officiel : [Automatic Differentiation with torch.autograd](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)

### 3.1 Un premier exemple

Prenons $y = x^2 + 2x$. En cours de mathématiques, on trouve la dérivée $\frac{dy}{dx} = 2x + 2$.
Pour $x = 3$, la dérivée vaut $2 \times 3 + 2 = 8$. Vérifions avec PyTorch.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)  # requires_grad=True: "track the operations on x"
y = x**2 + 2*x

y.backward()          # compute dy/dx
print(f"y      = {y.item()}")
print(f"dy/dx  = {x.grad.item()}   (expected: 2*3 + 2 = 8)")

### 3.2 Le graphe de calcul

Voici le plus simple réseau de neurones à une couche (exemple du tutoriel officiel) :
une entrée `x`, des paramètres `w` (poids) et `b` (biais), et une fonction de perte.

Pendant la **passe avant** (*forward*), PyTorch construit un **graphe** de toutes les opérations.
Pendant la **passe arrière** (*backward*), il parcourt ce graphe à l'envers pour calculer les gradients.
C'est la **rétropropagation** (*backpropagation*).

<p align="center"><img src="https://raw.githubusercontent.com/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/main/images/04_graphe_calcul.png" alt="Graphe de calcul : passe avant et passe arrière" width="900"></p>

In [ ]:
x = torch.ones(5)                           # input tensor
y = torch.zeros(3)                          # expected output
w = torch.randn(5, 3, requires_grad=True)   # parameter: weights
b = torch.randn(3, requires_grad=True)      # parameter: bias

z = torch.matmul(x, w) + b                  # forward pass
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

# Each tensor made by an operation remembers the function that made it
print(f"Gradient function for z    = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

In [ ]:
loss.backward()     # backward pass: fill w.grad and b.grad

print("w.grad (d loss / d w):\n", w.grad)
print("b.grad (d loss / d b):\n", b.grad)

### 3.3 Arrêter le suivi des gradients

Le suivi des opérations coûte de la mémoire et du temps. On l'arrête quand on n'entraîne pas :

- pendant l'**évaluation** ou l'**inférence** (on utilise le modèle sans le modifier) ;
- pour **geler** (*freeze*) des paramètres.

> **Lien avec le Jour 2 :** avec LoRA, on **gèle** le grand modèle pré-entraîné et on entraîne seulement quelques petits poids ajoutés.

In [ ]:
z = torch.matmul(x, w) + b
print("Normal          :", z.requires_grad)

with torch.no_grad():               # no tracking inside this block
    z = torch.matmul(x, w) + b
print("torch.no_grad() :", z.requires_grad)

z_det = (torch.matmul(x, w) + b).detach()   # a copy without history
print(".detach()       :", z_det.requires_grad)

### 3.4 Attention : les gradients s'additionnent

À chaque appel de `.backward()`, PyTorch **ajoute** les nouveaux gradients aux anciens dans `.grad`.
C'est pourquoi, dans la boucle d'entraînement, on remet les gradients à zéro à chaque étape (`optimizer.zero_grad()`).

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
for step in range(3):
    y = x**2              # dy/dx = 2x = 6
    y.backward()
    print(f"step {step}: x.grad = {x.grad.item()}")   # 6, then 12, then 18!

x.grad.zero_()            # reset to zero
print("after zero_():", x.grad.item())

### 3.5 La descente de gradient « à la main »

Le gradient indique la direction où la perte **monte**. Pour **réduire** la perte, on fait un petit pas dans la direction **opposée** :

$$w \leftarrow w - \text{lr} \times \frac{\partial \text{loss}}{\partial w}$$

`lr` est le **taux d'apprentissage** (*learning rate*) : la taille du pas.

<p align="center"><img src="https://raw.githubusercontent.com/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/main/images/05_descente_gradient.png" alt="Descente de gradient : descendre la vallée de la perte" width="900"></p>

Essayons sur un problème simple : retrouver la droite $y = 2x + 1$ à partir de points bruités.
Le modèle est $\hat{y} = w \cdot x + b$. Au départ, $w = 0$ et $b = 0$. Il doit **apprendre** $w \approx 2$ et $b \approx 1$.

In [ ]:
# Synthetic data: y = 2x + 1 + noise
torch.manual_seed(0)
X = torch.linspace(-1, 1, 100)
Y = 2 * X + 1 + 0.2 * torch.randn(100)

# Parameters to learn (start at 0)
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

lr = 0.1          # learning rate
losses = []

for step in range(100):
    Y_pred = w * X + b                       # 1. forward: prediction
    loss = ((Y_pred - Y) ** 2).mean()        # 2. loss: mean squared error
    loss.backward()                          # 3. backward: gradients
    with torch.no_grad():                    # 4. update (no tracking here)
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_()                           # 5. reset the gradients
    b.grad.zero_()
    losses.append(loss.item())
    if step % 20 == 0:
        print(f"step {step:3d} | loss {loss.item():.4f} | w {w.item():.3f} | b {b.item():.3f}")

print(f"\nLearned: w = {w.item():.3f} (true 2), b = {b.item():.3f} (true 1)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(X, Y, s=12, label="data")
axes[0].plot(X, (w * X + b).detach(), color="red", linewidth=2, label="learned line")
axes[0].set_title("Data and learned line")
axes[0].legend()
axes[1].plot(losses)
axes[1].set_title("Loss during training")
axes[1].set_xlabel("step")
axes[1].set_ylabel("loss (MSE)")
plt.tight_layout()
plt.show()

### ✍️ Exercice 2 : le taux d'apprentissage

Relancez la cellule de la descente de gradient avec `lr = 0.01`, puis avec `lr = 1.2`.
Que se passe-t-il ? Comparez avec le schéma ci-dessus.

<details>
<summary><b>Voir la réponse</b></summary>

- `lr = 0.01` : les pas sont trop petits. Après 100 étapes, `w` vaut environ 1 (au lieu de 2) et `b` environ 0,87 (au lieu de 1). Il faudrait beaucoup plus d'étapes.
- `lr = 1.2` : les pas sont trop grands. Le modèle « saute » par-dessus le minimum, de plus en plus loin. La perte explose (des nombres énormes).
- Le bon `lr` dépend du problème. On le choisit par essais. C'est un **hyperparamètre**.
</details>

## 4. Les données : `Dataset` et `DataLoader`

PyTorch sépare deux rôles :

- un **`Dataset`** stocke les exemples et leurs étiquettes (*labels*). Il répond à : « donne-moi l'exemple n° i » ;
- un **`DataLoader`** parcourt le `Dataset` par **petits lots** (*batches*), en mélangeant les exemples.

<p align="center"><img src="https://raw.githubusercontent.com/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/main/images/06_dataset_dataloader.png" alt="Dataset et DataLoader : des images aux lots" width="900"></p>

Nous utilisons **FashionMNIST** : 70 000 images de vêtements en niveaux de gris, de 28 × 28 pixels, réparties en 10 classes
(60 000 pour l'entraînement, 10 000 pour le test). C'est le jeu de données du tutoriel officiel.

Une **transformation** (*transform*) prépare chaque image : `v2.ToImage()` puis `v2.ToDtype(torch.float32, scale=True)`
transforment l'image en tenseur de nombres réels entre 0 et 1.

Tutoriels officiels : [Datasets & DataLoaders](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html) · [Transforms](https://docs.pytorch.org/tutorials/beginner/basics/transforms_tutorial.html)

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

# Image -> float tensor with values in [0, 1]
transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])

training_data = datasets.FashionMNIST(root="data", train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root="data", train=False, download=True, transform=transform)

print(f"Training images: {len(training_data)}")
print(f"Test images    : {len(test_data)}")

classes = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
classes_fr = ["T-shirt/haut", "Pantalon", "Pull", "Robe", "Manteau",
              "Sandale", "Chemise", "Basket", "Sac", "Bottine"]

Regardons un exemple, puis quelques images au hasard.

In [ ]:
image, label = training_data[0]          # a Dataset works like a list
print(f"Image shape : {image.shape}   (channels, height, width)")
print(f"Value range : {image.min().item():.1f} to {image.max().item():.1f}")
print(f"Label       : {label} -> {classes[label]} ({classes_fr[label]})")

figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3
for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(training_data), size=(1,)).item()
    img, label = training_data[sample_idx]
    figure.add_subplot(rows, cols, i)
    plt.title(f"{classes_fr[label]}")
    plt.axis("off")
    plt.imshow(img.squeeze(), cmap="gray")
plt.show()

Maintenant, le `DataLoader`. Pendant l'entraînement :

- on **mélange** (`shuffle=True`) les données à chaque époque, pour que le modèle n'apprenne pas l'ordre ;
- on donne les images par **lots de 64** : c'est plus rapide sur GPU et plus stable qu'une image à la fois.

In [ ]:
batch_size = 64

train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

X, y = next(iter(train_dataloader))      # take one batch
print(f"Shape of X [N, C, H, W]: {X.shape}")
print(f"Shape of y: {y.shape} {y.dtype}")
print(f"Number of batches per epoch: {len(train_dataloader)}")   # 60000 / 64 -> 938

**Et pour vos propres données ?** Il suffit d'écrire une classe avec deux méthodes :
`__len__` (combien d'exemples) et `__getitem__` (renvoyer l'exemple n° i).
Demain, nous utiliserons la même idée avec des phrases en swahili.

In [ ]:
from torch.utils.data import Dataset

class SquaresDataset(Dataset):
    """A tiny custom dataset: the input is x, the label is x squared."""
    def __init__(self, n):
        self.x = torch.arange(n, dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.x[idx] ** 2

ds = SquaresDataset(10)
print("len:", len(ds), "| item 3:", ds[3])
for xb, yb in DataLoader(ds, batch_size=4):
    print("batch:", xb, yb)

## 5. Construire le modèle

Dans PyTorch, un modèle est une classe qui hérite de **`nn.Module`**. Elle a deux parties :

- `__init__` : on **déclare** les couches ;
- `forward` : on dit **comment les données traversent** les couches.

Notre réseau transforme une image de 28 × 28 pixels en **10 scores**, un par classe.

<p align="center"><img src="https://raw.githubusercontent.com/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/main/images/07_architecture_modele.png" alt="Architecture du réseau : Flatten, Linear, ReLU, Linear, ReLU, Linear" width="900"></p>

- `nn.Flatten` met l'image « à plat » : 28 × 28 = 784 nombres ;
- `nn.Linear(entrées, sorties)` calcule `x @ W.T + b`. `W` et `b` sont des **paramètres appris** ;
- `nn.ReLU` est une **fonction d'activation** : elle garde les nombres positifs et remplace les négatifs par 0. Sans elle, plusieurs couches linéaires équivalent à une seule ;
- la sortie, ce sont 10 **logits** (des scores bruts). `softmax` les transforme en **probabilités** (positives, de somme 1).

Tutoriel officiel : [Build the Neural Network](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

In [ ]:
# How many parameters does the model learn?
total = 0
for name, param in model.named_parameters():
    print(f"{name:28s} {str(tuple(param.shape)):12s} {param.numel():>8,d}")
    total += param.numel()
print(f"{'TOTAL':41s} {total:>8,d}")

Suivons un lot de 3 images à travers chaque couche, et regardons la forme du tenseur.

In [ ]:
images = torch.rand(3, 1, 28, 28, device=device)   # 3 fake images
x = model.flatten(images)
print(f"{'input':22s} {tuple(images.shape)}")
print(f"{'Flatten':22s} {tuple(x.shape)}")
for layer in model.linear_relu_stack:
    x = layer(x)
    print(f"{layer.__class__.__name__:22s} {tuple(x.shape)}")

In [ ]:
# ReLU(x) = max(0, x)
xs = torch.linspace(-3, 3, 100)
plt.figure(figsize=(4, 3))
plt.plot(xs, nn.ReLU()(xs))
plt.title("ReLU")
plt.grid(alpha=0.3)
plt.show()

Le modèle n'est pas encore entraîné : ses poids sont aléatoires. Sa prédiction est donc un hasard.

In [ ]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)                          # call the model, not model.forward(X)
pred_probab = nn.Softmax(dim=1)(logits)    # logits -> probabilities
y_pred = pred_probab.argmax(1)

print("Logits        :", logits.detach().cpu().numpy().round(3))
print("Probabilities :", pred_probab.detach().cpu().numpy().round(3))
print(f"Predicted class: {classes_fr[y_pred.item()]} (random, the model has not learned yet)")

## 6. La boucle d'entraînement

Nous avons les données et le modèle. Il faut maintenant **entraîner** le modèle, c'est-à-dire ajuster ses paramètres.
Chaque tour de boucle fait 5 étapes :

<p align="center"><img src="https://raw.githubusercontent.com/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/main/images/08_boucle_entrainement.png" alt="La boucle d'entraînement en 5 étapes" width="700"></p>

Trois choix à faire :

- les **hyperparamètres** : `learning_rate` (taille du pas), `batch_size` (taille du lot), `epochs` (nombre de passages sur toutes les données) ;
- la **fonction de perte** : pour une classification, `nn.CrossEntropyLoss`. Elle mesure à quel point les logits sont loin de la bonne classe ;
- l'**optimiseur** : la règle de mise à jour des poids. Ici, la descente de gradient stochastique (`torch.optim.SGD`).

Tutoriel officiel : [Optimizing Model Parameters](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)

In [ ]:
learning_rate = 1e-3
epochs = 5

model = NeuralNetwork().to(device)   # a fresh model
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()                        # training mode (matters for dropout / batch norm)
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        pred = model(X)                  # 1-2. forward pass: prediction
        loss = loss_fn(pred, y)          # 3. loss

        loss.backward()                  # 4. backward pass: gradients
        optimizer.step()                 # 5. update the parameters
        optimizer.zero_grad()            #    reset the gradients

        if batch % 200 == 0:
            current = batch * dataloader.batch_size + len(X)
            print(f"  loss: {loss.item():>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()                         # evaluation mode
    test_loss, correct = 0, 0
    with torch.no_grad():                # no gradients needed to evaluate
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    accuracy = correct / size
    print(f"  Test: accuracy {100 * accuracy:>0.1f}%, avg loss {test_loss:>8f}")
    return test_loss, accuracy

In [ ]:
history = {"loss": [], "accuracy": []}
start = time.perf_counter()
for t in range(epochs):
    print(f"Epoch {t + 1}/{epochs}")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loss, accuracy = test_loop(test_dataloader, model, loss_fn)
    history["loss"].append(test_loss)
    history["accuracy"].append(accuracy)
print(f"Done in {time.perf_counter() - start:.0f} s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
epochs_axis = range(1, len(history["loss"]) + 1)
axes[0].plot(epochs_axis, history["loss"], marker="o")
axes[0].set_title("Test loss")
axes[0].set_xlabel("epoch")
axes[1].plot(epochs_axis, [100 * a for a in history["accuracy"]], marker="o", color="green")
axes[1].set_title("Test accuracy (%)")
axes[1].set_xlabel("epoch")
plt.tight_layout()
plt.show()

La perte baisse et la précision monte : le modèle **apprend**. Mais avec `SGD` et `lr = 1e-3`, il apprend lentement.

### ✍️ Exercice 3 : améliorer l'entraînement

Créez un **nouveau** modèle et entraînez-le 5 époques avec un autre optimiseur : `torch.optim.Adam(model.parameters(), lr=1e-3)`.
Comparez la précision finale avec celle de `SGD`.

Bonus : essayez aussi `SGD` avec `lr = 0.1`.

<details>
<summary><b>Voir la solution</b></summary>

```python
model_adam = NeuralNetwork().to(device)
optimizer_adam = torch.optim.Adam(model_adam.parameters(), lr=1e-3)
for t in range(5):
    print(f"Epoch {t + 1}")
    train_loop(train_dataloader, model_adam, loss_fn, optimizer_adam)
    test_loop(test_dataloader, model_adam, loss_fn)
```

Dans notre test, Adam atteint environ 85 % après 1 époque et 87 % après 3 époques (contre environ 64 % pour SGD avec `lr = 1e-3`).
`SGD` avec `lr = 0.1` atteint environ 80 %, mais la précision monte et descend d'une époque à l'autre.
Adam adapte la taille du pas pour chaque paramètre. C'est l'optimiseur le plus utilisé aujourd'hui (avec sa variante `AdamW`, que nous utiliserons demain).
</details>

In [ ]:
# Your code here (Exercise 3)

## 7. Sauvegarder et charger un modèle

Un modèle entraîné a de la valeur : on ne veut pas l'entraîner à nouveau à chaque fois.
PyTorch sauvegarde les **poids** du modèle dans un dictionnaire appelé **`state_dict`**.

<p align="center"><img src="https://raw.githubusercontent.com/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/main/images/09_sauvegarde_chargement.png" alt="Sauvegarder et charger un modèle avec state_dict" width="900"></p>

Tutoriel officiel : [Save and Load the Model](https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html)

In [ ]:
# What is inside the state_dict?
for key, value in model.state_dict().items():
    print(f"{key:28s} {tuple(value.shape)}")

torch.save(model.state_dict(), "model.pth")
import os
print(f"\nSaved model.pth ({os.path.getsize('model.pth') / 1e6:.1f} MB)")

In [ ]:
# Load: create the same architecture, then load the weights into it
loaded_model = NeuralNetwork().to(device)
loaded_model.load_state_dict(torch.load("model.pth", weights_only=True))
loaded_model.eval()     # evaluation mode before predictions

# Predict 8 test images
fig = plt.figure(figsize=(12, 3.5))
with torch.no_grad():
    for i in range(8):
        x, y = test_data[i]
        pred = loaded_model(x.unsqueeze(0).to(device)).argmax(1).item()
        ax = fig.add_subplot(1, 8, i + 1)
        ax.imshow(x.squeeze(), cmap="gray")
        ax.axis("off")
        color = "green" if pred == y else "red"
        ax.set_title(f"{classes_fr[pred]}\n(vrai : {classes_fr[y]})", color=color, fontsize=9)
plt.tight_layout()
plt.show()

> **À savoir**
> - `weights_only=True` charge seulement des tenseurs. C'est plus **sûr** : un fichier `.pth` inconnu ne peut pas exécuter de code caché.
> - On peut aussi sauvegarder le modèle entier avec `torch.save(model, "model.pth")`, mais le chargement demande alors `weights_only=False`. Préférez le `state_dict`.
> - Dans Colab, les fichiers sont effacés à la fin de la session. Pour garder `model.pth` : `from google.colab import files; files.download("model.pth")`.

## 8. Pont vers le Jour 2 : les modèles de langage

Demain, nous allons adapter (*fine-tuner*) un vrai petit **modèle de langage** à une langue africaine.
Un modèle de langage fait une chose simple : **prédire le prochain morceau de texte** (le prochain *token*).
En répétant cette prédiction, il écrit des phrases.

<p align="center"><img src="https://raw.githubusercontent.com/ganji759/IndabaX-DRC-2026-PyTorch-Workshop/main/images/10_modele_de_langage.png" alt="Comment un modèle de langage lit et écrit du texte" width="900"></p>

1. Le **tokenizer** découpe le texte en **tokens** (des mots ou des morceaux de mots) et donne un **identifiant** (un nombre) à chacun.
2. Une couche **`nn.Embedding`** transforme chaque identifiant en un **vecteur** de nombres.
3. Le **modèle** (un *Transformer*) calcule des **probabilités** pour le prochain token.

### 8.1 Le tokenizer

Utilisons le tokenizer d'un vrai petit modèle, `SmolLM2-135M-Instruct` (le modèle de l'exemple officiel ExecuTorch).
Comparons la même phrase en anglais, en français, en swahili et en lingala.

In [ ]:
# The tokenizer is small (a few MB). Colab already has the transformers library.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

sentences = {
    "English": "Good morning, my friend. How are you?",
    "Français": "Bonjour mon ami. Comment vas-tu ?",
    "Kiswahili": "Habari ya asubuhi, rafiki yangu. Habari yako?",
    "Lingala": "Mbote moninga na ngai. Ozali malamu?",
}

for language, text in sentences.items():
    ids = tokenizer.encode(text)
    print(f"{language:10s} {len(ids):3d} tokens")

print()
ids = tokenizer.encode(sentences["Kiswahili"])
print("IDs    :", ids)
print("Tokens :", tokenizer.convert_ids_to_tokens(ids))
print("Decode :", tokenizer.decode(ids))

**Observation :** pour dire la même chose, le swahili et le lingala demandent **plus de tokens** que l'anglais.
Ce tokenizer a été construit surtout avec des textes en anglais. Il découpe nos langues en petits morceaux.
Conséquence : le modèle « comprend » moins bien ces langues, et il est plus lent et plus cher pour elles.

C'est une des raisons pour lesquelles nous utiliserons demain un modèle déjà adapté à des langues africaines,
puis nous le fine-tunerons sur des instructions en swahili.

### 8.2 `nn.Embedding` : des identifiants aux vecteurs

Une couche `nn.Embedding` est une **table** : une ligne de nombres (un vecteur) par identifiant.
Ces vecteurs sont des **paramètres appris**, comme les poids d'une couche `nn.Linear`.

In [ ]:
vocab_size = tokenizer.vocab_size
embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=8)

ids_tensor = torch.tensor(ids)
vectors = embedding(ids_tensor)
print(f"Vocabulary size : {vocab_size}")
print(f"IDs shape       : {tuple(ids_tensor.shape)}")
print(f"Vectors shape   : {tuple(vectors.shape)}   (one vector of 8 numbers per token)")
print("Learnable       :", embedding.weight.requires_grad)

### 8.3 Un mini modèle de langage, entraîné avec **la même boucle**

Pour finir, entraînons un tout petit modèle de langage sur un court texte en swahili (écrit pour cet atelier).
Il travaille **caractère par caractère** : en voyant un caractère, il prédit le suivant.
C'est un modèle **bigramme**. Il contient une seule couche `nn.Embedding`.

Regardez bien la boucle d'entraînement : ce sont les **mêmes étapes** que pour FashionMNIST.

In [ ]:
text = (
    "Habari ya asubuhi. Habari njema. Jina langu ni Amani. Ninaishi Lubumbashi. "
    "Ninapenda kusoma na kujifunza. Leo tunajifunza akili bandia na PyTorch. "
    "Kompyuta inajifunza kutoka kwenye data. Asante sana rafiki yangu. "
    "Karibu sana. Tutaonana kesho. "
)

chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}   # character -> ID (a tiny tokenizer)
itos = {i: ch for ch, i in stoi.items()}       # ID -> character

data = torch.tensor([stoi[c] for c in text])
X_chars, Y_chars = data[:-1], data[1:]         # input: a character, label: the next one
print(f"{len(text)} characters, vocabulary of {V} characters")
print("First pairs:", [(itos[a.item()], itos[b.item()]) for a, b in zip(X_chars[:6], Y_chars[:6])])

In [ ]:
class BigramModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # One row per character: the scores (logits) of the next character
        self.table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx):
        return self.table(idx)

    @torch.no_grad()
    def generate(self, start, length=150):
        idx = torch.tensor([stoi[start]])
        out = start
        for _ in range(length):
            probs = torch.softmax(self(idx), dim=-1)            # next-character probabilities
            idx = torch.multinomial(probs, num_samples=1)[0]    # pick one at random
            out += itos[idx.item()]
        return out

torch.manual_seed(42)
lm = BigramModel(V)
print(f"Parameters: {sum(p.numel() for p in lm.parameters())}  ({V} x {V})")
print("Before training:", lm.generate("H"))

In [ ]:
optimizer_lm = torch.optim.Adam(lm.parameters(), lr=0.1)
loss_fn_lm = nn.CrossEntropyLoss()

for step in range(301):
    logits = lm(X_chars)                   # 1-2. forward pass
    loss = loss_fn_lm(logits, Y_chars)     # 3. loss
    loss.backward()                        # 4. gradients
    optimizer_lm.step()                    # 5. update
    optimizer_lm.zero_grad()
    if step % 100 == 0:
        print(f"step {step:3d} | loss {loss.item():.3f}")

torch.manual_seed(1)
print("\nAfter training:", lm.generate("H"))

Le texte généré **ressemble** un peu au swahili (syllabes, espaces, points), mais il n'a pas de sens.
Normal : ce modèle ne voit qu'**un seul caractère** pour prédire le suivant.

Un vrai modèle de langage suit exactement le même principe, avec trois grandes différences :

| | Notre mini modèle | Un vrai modèle (demain) |
|---|---|---|
| Unité | caractère | token (morceau de mot) |
| Contexte | 1 caractère | des milliers de tokens (grâce au *Transformer*) |
| Paramètres | environ 1 000 | des centaines de millions ou des milliards |
| Données | 1 paragraphe | des milliards de mots |

**Demain**, nous prendrons un modèle déjà pré-entraîné, et nous l'adapterons à des instructions en swahili avec **LoRA**,
sur le même GPU gratuit de Colab. La boucle d'entraînement sera la même que celle d'aujourd'hui.

## 9. Résumé

| Concept | Ce qu'il faut retenir | Code |
|---|---|---|
| Tenseur | tableau de nombres à N dimensions | `torch.tensor`, `.shape`, `.dtype`, `.device` |
| GPU | calcul parallèle, beaucoup plus rapide | `.to(device)` |
| Autograd | calcul automatique des gradients | `requires_grad=True`, `loss.backward()`, `.grad` |
| Données | exemples par lots, mélangés | `Dataset`, `DataLoader(batch_size, shuffle)` |
| Modèle | couches + passe avant | `nn.Module`, `__init__`, `forward` |
| Entraînement | prédire, mesurer l'erreur, corriger | `loss_fn`, `optimizer.step()`, `optimizer.zero_grad()` |
| Évaluation | sans gradients, mode évaluation | `model.eval()`, `torch.no_grad()` |
| Sauvegarde | on sauvegarde les poids | `torch.save(model.state_dict())`, `load_state_dict` |
| Modèle de langage | prédire le prochain token | tokenizer, `nn.Embedding` |

### La boucle d'entraînement à retenir

```python
for epoch in range(epochs):
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        pred = model(X)              # forward
        loss = loss_fn(pred, y)      # loss
        loss.backward()              # gradients
        optimizer.step()             # update
        optimizer.zero_grad()        # reset
```

### Pour aller plus loin

- [Learn the Basics](https://docs.pytorch.org/tutorials/beginner/basics/intro.html) : le parcours officiel complet (en anglais), avec un lien « Run in Google Colab » sur chaque page.
- [Deep Learning with PyTorch: A 60 Minute Blitz](https://docs.pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html)
- [Introduction to PyTorch on YouTube](https://docs.pytorch.org/tutorials/beginner/introyt/introyt_index.html)
- [Documentation de PyTorch](https://docs.pytorch.org/docs/stable/index.html)
- [Forum PyTorch](https://discuss.pytorch.org/) : pour poser vos questions.

### Exercices bonus (à la maison)

1. Ajoutez une couche `nn.Dropout(0.2)` après chaque `ReLU`. La précision de test change-t-elle ?
2. Entraînez le modèle 10 époques avec Adam. À partir de quelle époque la précision n'augmente presque plus ?
3. Remplacez FashionMNIST par `datasets.MNIST` (chiffres écrits à la main). Que faut-il changer dans le code ?
4. Ajoutez des phrases en swahili (ou en lingala, en kikongo, en tshiluba) dans le texte du mini modèle de langage. Le texte généré change-t-il ?

---

**Crédits.** Ce notebook adapte les tutoriels officiels [PyTorch Learn the Basics](https://github.com/pytorch/tutorials/tree/main/beginner_source/basics)
(licence BSD-3-Clause, © PyTorch contributors). Données : [FashionMNIST](https://github.com/zalandoresearch/fashion-mnist) (Zalando Research, licence MIT).
Schémas faits avec [Excalidraw](https://excalidraw.com). Les fichiers sources des schémas sont dans le dossier `images/excalidraw/` du dépôt.

**Rendez-vous demain pour le Jour 2 : fine-tuning, inférence et déploiement d'un petit modèle de langage.** 🚀